In [ ]:
!python "C:\Program Files (x86)\Eclipse\Sumo\tools\district\gridDistricts.py" -n ../map.net.xml -o ../grids.taz.xml -w 500

In [3]:
!netedit  -c sim.sumocfg

^C


In [13]:
import xml.etree.ElementTree as ET
import random

taz_file = "../grids.taz.xml"
output_file = "od.xml"

begin_time = 0
end_time = 3600

min_count = 10      
max_count = 100     

seed = 42
random.seed(seed)

# -----------------------------
# LOAD TAZ IDs
# -----------------------------
tree = ET.parse(taz_file)
root = tree.getroot()

taz_ids = []
for taz in root.findall("taz"):
    taz_ids.append(taz.get("id"))

print("TAZ count:", len(taz_ids))

# -----------------------------
# BUILD OD MATRIX (ALL-TO-ALL)
# -----------------------------
relations = []

for origin in taz_ids:
    for dest in taz_ids:
        if origin == dest:
            continue

        count = random.randint(min_count, max_count)

        relations.append({
            "from": origin,
            "to": dest,
            "count": count
        })

# -----------------------------
# WRITE od.xml
# -----------------------------
with open(output_file, "w", encoding="utf-8") as f:
    f.write("<?xml version='1.0' encoding='utf-8'?>\n")
    f.write("<data>\n")
    f.write(f'    <interval begin="{begin_time}" end="{end_time}">\n')

    for r in relations:
        f.write(
            f'        <tazRelation from="{r["from"]}" to="{r["to"]}" count="{r["count"]}" />\n'
        )

    f.write("    </interval>\n")
    f.write("</data>\n")

print("OD file created:", output_file)

TAZ count: 12
OD file created: od.xml


In [22]:
import xml.etree.ElementTree as ET
import numpy as np

tree = ET.parse("od.xml")
root = tree.getroot()

relations = root.findall(".//tazRelation")

# Collect all TAZ IDs
tazs = sorted({
    r.attrib["from"] for r in relations
}.union({
    r.attrib["to"] for r in relations
}))

taz_to_idx = {taz: i for i, taz in enumerate(tazs)}

# Create matrix
od = np.zeros((len(tazs), len(tazs)), dtype=int)

for r in relations:
    i = taz_to_idx[r.attrib["from"]]
    j = taz_to_idx[r.attrib["to"]]
    od[i, j] = int(r.attrib["count"])

print(tazs)
print(od)

['0_3', '1_1', '1_2', '1_3', '2_0', '2_1', '2_2', '3_0', '3_1', '3_2', '4_1', '4_2']
[[  0  91  24  13  45  41  38  27  23  96  79  21]
 [ 85   0  64  14  13  21  37  39  74  87  13  81]
 [ 35  93   0  99  79  63  38  67  85  45  10  30]
 [ 99  64  53   0  45  29  37  53  23  21  58  22]
 [ 55  54  87  43   0  15  68  78  25  58  20  80]
 [ 47  90  89  56  83   0  34 100  18  15  94  39]
 [ 47  20  39  22  58  45   0  68  91  56  30  57]
 [ 55  36  95  44  99  97  92   0  19  87  91  31]
 [ 78  41  30  69  58  44  91  98   0  81  38  97]
 [ 51  17  39  14  50  61  44  18  37   0  82  50]
 [ 37  93  73  60  92  68  28  43  27  41   0  81]
 [ 78  43  84  64  84  61  56  38  27  75  73   0]]


In [14]:
!od2trips -z od.xml -n ../grids.taz.xml -o trips.xml

Parsing time 0.00
Parsing time 1.84
Parsing time 4.33
Parsing time 6.04
Parsing time 7.63
Parsing time 8.70
Parsing time 10.87
Parsing time 12.17
Parsing time 14.07
Parsing time 15.55
Parsing time 16.72
Parsing time 18.30
Parsing time 19.61
Parsing time 21.21
Parsing time 22.42
Parsing time 23.87
Parsing time 25.12
Parsing time 26.95
Parsing time 28.04
Parsing time 29.39
Parsing time 31.39
Parsing time 32.84
Parsing time 34.38
Parsing time 35.81
Parsing time 37.22
Parsing time 38.76
Parsing time 42.04
Parsing time 43.31
Parsing time 44.99
Parsing time 46.74
Parsing time 47.95
Parsing time 49.05
Parsing time 50.39
Parsing time 53.26
Parsing time 54.38
Parsing time 55.39
Parsing time 57.81
Parsing time 58.83
Parsing time 59.85
Parsing time 62.31
Parsing time 63.61
Parsing time 65.23
Parsing time 66.73
Parsing time 68.70
Parsing time 69.75
Parsing time 70.96
Parsing time 72.15
Parsing time 73.45
Parsing time 74.54
Parsing time 76.46
Parsing time 77.48
Parsing time 80.18
Parsing time 81.49

In [15]:
!duarouter -n ../map.net.xml -r trips.xml -o routes.rou.xml --ignore-errors true

Reading up to time step: 0.79
Reading up to time step: 200.79
Reading up to time step: 400.79
Reading up to time step: 600.79
Reading up to time step: 800.79
Reading up to time step: 1000.79
Reading up to time step: 1200.79
Reading up to time step: 1400.79
Reading up to time step: 1600.79
Reading up to time step: 1800.79
Reading up to time step: 2000.79
Reading up to time step: 2200.79
Reading up to time step: 2400.79
Reading up to time step: 2600.79
Reading up to time step: 2800.79
Reading up to time step: 3000.79
Reading up to time step: 3200.79
Reading up to time step: 3400.79
Reading up to time step: 3600.79
Success.


In [16]:
import os
import json
import xml.etree.ElementTree as ET

In [17]:
def run_sumo():
    cmd = "sumo -c sim.sumocfg"
    result = os.system(cmd)

    if result != 0:
        print("❌ SUMO failed")
    else:
        print("✅ SUMO finished")

run_sumo()

✅ SUMO finished


In [18]:
def extract_edge_travel_times(file_path="edges.xml"):
    tree = ET.parse(file_path)
    root = tree.getroot()

    edge_weights = {}

    for interval in root.findall("interval"):
        for edge in interval.findall("edge"):
            edge_id = edge.get("id")

            # SUMO sometimes uses traveltime or meanTravelTime
            # tt = edge.get("traveltime") or edge.get("meanTravelTime")
            tt = edge.get("speedRelative")

            if edge_id is not None and tt is not None:
                edge_weights[edge_id] = float(tt)

    return edge_weights


edge_weights = extract_edge_travel_times()

print("Edges loaded:", len(edge_weights))
print(list(edge_weights.items())[:5])

Edges loaded: 360
[('-1035578874', 0.73), ('-1035578887#0', 0.23), ('-1035578887#1', 0.9), ('-1035578887#2', 0.88), ('-1199956439', 0.12)]


In [19]:
def save_neshan_format(edge_weights, output_path="../data/neshan-sample-weights.json"):
    with open(output_path, "w") as f:
        json.dump(edge_weights, f, indent=2)

    print(f"💾 Saved {len(edge_weights)} edges to {output_path}")


save_neshan_format(edge_weights)

💾 Saved 360 edges to ../data/neshan-sample-weights.json


In [20]:
# !python "C:\Program Files (x86)\Eclipse\Sumo\tools\visualization\plot_net_dump.py" \
#   -n ../map.net.xml \
#   -i ../edges.xml \
#   --measures speedRelative,speedRelative \
#   --default-width 1.0 \
#   --default-color "#606060" \
#   --min-color-value 0.0 \
#   --max-color-value 1.3 \
#   --colormap "#0:#0000c0,.5:#808080,1:#c00000" \
#   -o result_full.png \
#   --verbose

In [21]:
import xml.etree.ElementTree as ET

# Load the file
tree = ET.parse('edges.xml')
root = tree.getroot()

data = []

for interval in root.findall('interval'):
    for edge in interval.findall('edge'):
        edge_id = edge.get('id')
        speed_rel = float(edge.get('speedRelative', 1.0))   # default 1.0 if missing
        speed = float(edge.get('speed', 0))
        sampledSeconds = float(edge.get('sampledSeconds', 0))
        entered = float(edge.get('entered', 0))
        flow = float(edge.get('flow', 0))
        
        data.append({
            'id': edge_id,
            'speedRelative': speed_rel,
            'speed': speed,
            'sampledSeconds': sampledSeconds,
            'entered': entered,
            'flow': flow
        })

# Sort by slowest (lowest speedRelative) first
top_10_slowest = sorted(data, key=lambda x: x['speedRelative'])[:10]
top_10_fastest = sorted(data, key=lambda x: x['speedRelative'], reverse=True)[:10]

print("=== TOP 10 SLOWEST EDGES (by speedRelative) ===")
for i, e in enumerate(top_10_slowest, 1):
    print(f"{i:2d}. Edge {e['id']:25} | "
          f"speedRelative: {e['speedRelative']:.3f} | "
          f"Speed: {e['speed']:.2f} m/s | "
          f"Flow: {e['flow']:.1f} | "
          f"Entered: {e['entered']:.0f}")

print("\n=== TOP 10 FASTEST EDGES (by speedRelative) ===")
for i, e in enumerate(top_10_fastest, 1):
    print(f"{i:2d}. Edge {e['id']:25} | "
          f"speedRelative: {e['speedRelative']:.3f} | "
          f"Speed: {e['speed']:.2f} m/s | "
          f"Flow: {e['flow']:.1f} | "
          f"Entered: {e['entered']:.0f}")

=== TOP 10 SLOWEST EDGES (by speedRelative) ===
 1. Edge -227547757#0              | speedRelative: 0.000 | Speed: 0.03 m/s | Flow: 0.8 | Entered: 29
 2. Edge 1035578885#1              | speedRelative: 0.000 | Speed: 0.03 m/s | Flow: 0.7 | Entered: 12
 3. Edge -1461800851               | speedRelative: 0.010 | Speed: 0.08 m/s | Flow: 0.2 | Entered: 1
 4. Edge -489251765                | speedRelative: 0.010 | Speed: 0.12 m/s | Flow: 15.5 | Entered: 12
 5. Edge 1199956439                | speedRelative: 0.010 | Speed: 0.18 m/s | Flow: 9.0 | Entered: 11
 6. Edge 425989942#6               | speedRelative: 0.010 | Speed: 0.18 m/s | Flow: 98.5 | Entered: 107
 7. Edge 425989942#7               | speedRelative: 0.010 | Speed: 0.11 m/s | Flow: 68.9 | Entered: 76
 8. Edge 425989943                 | speedRelative: 0.010 | Speed: 0.26 m/s | Flow: 91.0 | Entered: 119
 9. Edge 4337761#1                 | speedRelative: 0.010 | Speed: 0.12 m/s | Flow: 54.0 | Entered: 62
10. Edge 489251769#1        